# Busca dos parâmetros

Testamos 20 configurações com 10 seeds cada (0 a 9). Cada treino tem 600 episódios. Reservamos as seeds 100 a 109 para a avaliação final. As grades são 10×10 e 30×30; para cada algoritmo, escolhemos a configuração com melhor recompensa média nos últimos 100 episódios.

In [1]:
%pip -q install gymnasium==1.2.3 numpy==2.2.6 pandas==2.3.3 numba==0.61.2 matplotlib==3.10.7 seaborn==0.13.2

Note: you may need to restart the kernel to use updated packages.


C:\Users\Admin\Documents\Reinforcement-Learning\.venv\Scripts\python.exe: No module named pip


In [2]:
import numpy as np

def estado_discreto(obs, grade):
    pos = max(0, min(grade-1, int((obs[0] + 1.2) / 1.8 * grade)))
    vel = max(0, min(grade-1, int((obs[1] + .07) / .14 * grade)))
    return pos * grade + vel

In [3]:
from numba import njit

@njit
def iniciar_seed(seed):
    np.random.seed(seed)

@njit
def escolher_acao(q, s, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(3)
    melhores = np.flatnonzero(q[s] == q[s].max())
    return melhores[np.random.randint(len(melhores))]

@njit
def atualizar_q(q, s, a, r, ns, terminou, alpha, gamma):
    alvo = r if terminou else r + gamma * q[ns].max()
    q[s, a] += alpha * (alvo - q[s, a])

@njit
def atualizar_sarsa(q, tracos, s, a, r, ns, na, terminou, alpha, gamma, lam):
    alvo = r if terminou else r + gamma * q[ns, na]
    erro = alvo - q[s, a]
    tracos[s, a] += 1
    atualizacoes = 0
    for i in range(q.shape[0]):
        for j in range(3):
            if tracos[i, j] != 0:
                q[i, j] += alpha * erro * tracos[i, j]
                tracos[i, j] *= gamma * lam
                atualizacoes += 1
    return atualizacoes

@njit
def atualizar_dyna(q, s, a, r, ns, terminou, alpha, gamma, n,
                   recompensas, proximos, terminais, vistos, pares, quantidade):
    atualizar_q(q, s, a, r, ns, terminou, alpha, gamma)
    par = s * 3 + a
    if not vistos[par]:
        vistos[par] = True
        pares[quantidade] = par
        quantidade += 1
    recompensas[par], proximos[par], terminais[par] = r, ns, terminou
    for _ in range(n):
        par = pares[np.random.randint(quantidade)]
        ss, aa = par // 3, par % 3
        atualizar_q(q, ss, aa, recompensas[par], proximos[par],
                    terminais[par], alpha, gamma)
    return quantidade

In [4]:
import time
from pathlib import Path
import gymnasium as gym
import pandas as pd

EPISODIOS = 600
PASTA = Path('resultados_comparacao')

def configuracoes():
    opcoes = []
    perfis = [(0.1, 0.99, 0.99), (0.3, 0.999, 0.995)]
    for grade in (10, 30):
        for p, (alpha, gamma, decaimento) in enumerate(perfis):
            base = dict(grid=grade, alpha=alpha, gamma=gamma,
                        epsilon_initial=1.0, epsilon_min=0.05,
                        epsilon_decay=decaimento)
            opcoes.append(dict(base, config_id=f'q_g{grade}_p{p}',
                               algorithm='Q-learning', planning_steps=0, **{'lambda': 0.0}))
            for lam in (0.5, 0.9):
                opcoes.append(dict(base, config_id=f'sarsa_g{grade}_p{p}_l{int(lam*10)}',
                                   algorithm='SARSA(lambda)', planning_steps=0,
                                   **{'lambda': lam}))
            for n in (5, 20):
                opcoes.append(dict(base, config_id=f'dyna_g{grade}_p{p}_n{n}',
                                   algorithm='Dyna-Q', planning_steps=n, **{'lambda': 0.0}))
    return opcoes

def treinar(config, seed, fase):
    grade, algoritmo = config['grid'], config['algorithm']
    alpha, gamma = config['alpha'], config['gamma']
    q = np.zeros((grade * grade, 3))
    tracos = np.zeros_like(q)
    tamanho = q.size
    recompensas = np.zeros(tamanho)
    proximos = np.zeros(tamanho, dtype=np.int64)
    terminais = np.zeros(tamanho, dtype=np.bool_)
    vistos = np.zeros(tamanho, dtype=np.bool_)
    pares = np.zeros(tamanho, dtype=np.int64)
    quantidade = 0

    # A primeira chamada compila as funções. Reiniciamos a seed antes do treino.
    iniciar_seed(0)
    escolher_acao(q, 0, 0.0)
    if algoritmo == 'SARSA(lambda)':
        atualizar_sarsa(q.copy(), tracos.copy(), 0, 0, -1.0, 1, 1,
                        False, alpha, gamma, config['lambda'])
    elif algoritmo == 'Dyna-Q':
        atualizar_dyna(q.copy(), 0, 0, -1.0, 1, False, alpha, gamma,
                       config['planning_steps'], recompensas.copy(), proximos.copy(),
                       terminais.copy(), vistos.copy(), pares.copy(), 0)
    else:
        atualizar_q(q.copy(), 0, 0, -1.0, 1, False, alpha, gamma)
    iniciar_seed(seed)

    ambiente = gym.make('MountainCar-v0')
    ambiente.action_space.seed(seed)
    registros, passos_totais, atualizacoes_totais = [], 0, 0
    inicio = time.perf_counter()
    try:
        for episodio in range(EPISODIOS):
            obs, _ = ambiente.reset(seed=seed if episodio == 0 else None)
            tracos.fill(0)
            epsilon = max(config['epsilon_min'],
                          config['epsilon_initial'] * config['epsilon_decay'] ** episodio)
            estado = estado_discreto(obs, grade)
            acao = escolher_acao(q, estado, epsilon)
            recompensa_total, passos, atualizacoes = 0, 0, 0
            while True:
                obs, recompensa, terminou, truncou, _ = ambiente.step(acao)
                proximo = estado_discreto(obs, grade)
                if algoritmo == 'SARSA(lambda)':
                    na = escolher_acao(q, proximo, epsilon) if not terminou else 0
                    atualizacoes += atualizar_sarsa(q, tracos, estado, acao, recompensa,
                        proximo, na, terminou, alpha, gamma, config['lambda'])
                elif algoritmo == 'Dyna-Q':
                    quantidade = atualizar_dyna(q, estado, acao, recompensa, proximo,
                        terminou, alpha, gamma, config['planning_steps'], recompensas,
                        proximos, terminais, vistos, pares, quantidade)
                    atualizacoes += 1 + config['planning_steps']
                else:
                    atualizar_q(q, estado, acao, recompensa, proximo,
                                terminou, alpha, gamma)
                    atualizacoes += 1
                recompensa_total += recompensa
                passos += 1
                passos_totais += 1
                if terminou or truncou:
                    break
                estado = proximo
                acao = na if algoritmo == 'SARSA(lambda)' else escolher_acao(q, estado, epsilon)
            atualizacoes_totais += atualizacoes
            registros.append(dict(**config, phase=fase, seed=seed, episode=episodio+1,
                                  episode_reward=recompensa_total, episode_steps=passos,
                                  cumulative_steps=passos_totais,
                                  episode_q_updates=atualizacoes, q_updates=atualizacoes_totais,
                                  success=bool(terminou), epsilon=epsilon,
                                  elapsed_seconds=time.perf_counter()-inicio))
    finally:
        ambiente.close()
    return pd.DataFrame(registros), q

def rodar(configs, seeds, fase):
    PASTA.mkdir(exist_ok=True)
    dados = []
    for config in configs:
        for seed in seeds:
            arquivo = PASTA / f'{fase}_{config["config_id"]}_seed{seed}.csv'
            tabela_q = PASTA / f'{config["config_id"]}_seed{seed}_q.npy'
            if arquivo.exists() and (fase == 'busca' or tabela_q.exists()):
                resultado = pd.read_csv(arquivo)
            else:
                resultado, q = treinar(config, seed, fase)
                resultado.to_csv(arquivo, index=False)
                if fase == 'avaliacao':
                    np.save(tabela_q, q)
            dados.append(resultado)
        print(config['config_id'], 'concluído')
    dados = pd.concat(dados, ignore_index=True)
    dados.to_csv(PASTA / f'{fase}.csv', index=False)
    return dados

def resumo(dados):
    ultimos = dados[dados.episode > EPISODIOS - 100]
    chaves = ['config_id', 'algorithm', 'grid', 'seed']
    por_seed = ultimos.groupby(chaves, as_index=False).agg(
        final_reward=('episode_reward', 'mean'), success_rate=('success', 'mean'))
    custo = dados.groupby(chaves, as_index=False).agg(
        total_q_updates=('q_updates', 'max'), total_steps=('cumulative_steps', 'max'))
    por_seed = por_seed.merge(custo, on=chaves)
    return por_seed.groupby(chaves[:-1], as_index=False).agg(
        mean_final_reward=('final_reward', 'mean'),
        std_final_reward=('final_reward', 'std'),
        mean_success_rate=('success_rate', 'mean'),
        mean_q_updates=('total_q_updates', 'mean'),
        mean_steps=('total_steps', 'mean'),
        seeds=('seed', 'nunique')).sort_values(
            ['mean_final_reward', 'mean_q_updates'], ascending=[False, True])

In [5]:
import json

busca = rodar(configuracoes(), range(10), 'busca')
ranking = resumo(busca)
ranking.to_csv(PASTA / 'ranking_busca.csv', index=False)
ids = set(ranking.drop_duplicates(['algorithm', 'grid']).config_id)
melhores = [c for c in configuracoes() if c['config_id'] in ids]
(PASTA / 'melhores_parametros.json').write_text(
    json.dumps(melhores, indent=2, ensure_ascii=False), encoding='utf-8')
display(ranking)
display(pd.DataFrame(melhores))

q_g10_p0 concluído
sarsa_g10_p0_l5 concluído
sarsa_g10_p0_l9 concluído
dyna_g10_p0_n5 concluído
dyna_g10_p0_n20 concluído


q_g10_p1 concluído
sarsa_g10_p1_l5 concluído
sarsa_g10_p1_l9 concluído
dyna_g10_p1_n5 concluído
dyna_g10_p1_n20 concluído
q_g30_p0 concluído


sarsa_g30_p0_l5 concluído
sarsa_g30_p0_l9 concluído
dyna_g30_p0_n5 concluído
dyna_g30_p0_n20 concluído
q_g30_p1 concluído
sarsa_g30_p1_l5 concluído


sarsa_g30_p1_l9 concluído
dyna_g30_p1_n5 concluído
dyna_g30_p1_n20 concluído


,config_id,algorithm,grid,mean_final_reward,std_final_reward,mean_success_rate,mean_q_updates,mean_steps,seeds
13,sarsa_g10_p0_l9,SARSA(lambda),10,-161.867,10.459256,0.825,2313902.6,105594.6,10
12,sarsa_g10_p0_l5,SARSA(lambda),10,-179.573,8.082584,0.627,2634348.0,113831.3,10
19,sarsa_g30_p1_l9,SARSA(lambda),30,-187.341,13.880052,0.397,6488353.4,118580.5,10
14,sarsa_g10_p1_l5,SARSA(lambda),10,-191.459,8.424188,0.285,1934659.9,118395.8,10
5,dyna_g30_p0_n5,Dyna-Q,30,-193.133,2.788915,0.272,706335.0,117722.5,10
15,sarsa_g10_p1_l9,SARSA(lambda),10,-194.578,3.167038,0.198,1977080.4,119153.1,10
8,q_g10_p0,Q-learning,10,-194.786,3.764835,0.203,118396.0,118396.0,10
18,sarsa_g30_p1_l5,SARSA(lambda),30,-196.566,1.013982,0.153,6946216.9,119431.4,10
9,q_g10_p1,Q-learning,10,-198.192,1.830002,0.071,119708.0,119708.0,10
4,dyna_g30_p0_n20,Dyna-Q,30,-199.267,0.683879,0.042,2513095.2,119671.2,10


,grid,alpha,gamma,epsilon_initial,epsilon_min,epsilon_decay,config_id,algorithm,planning_steps,lambda
0,10,0.1,0.990,1.0,0.05,0.990,q_g10_p0,Q-learning,0,0.0
1,10,0.1,0.990,1.0,0.05,0.990,sarsa_g10_p0_l9,SARSA(lambda),0,0.9
2,10,0.1,0.990,1.0,0.05,0.990,dyna_g10_p0_n5,Dyna-Q,5,0.0
3,30,0.1,0.990,1.0,0.05,0.990,dyna_g30_p0_n5,Dyna-Q,5,0.0
4,30,0.3,0.999,1.0,0.05,0.995,q_g30_p1,Q-learning,0,0.0
5,30,0.3,0.999,1.0,0.05,0.995,sarsa_g30_p1_l9,SARSA(lambda),0,0.9
